In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, os, math, gc

# Model link: https://www.kaggle.com/datasets/saisaaho/v10-models


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATA_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
if not os.path.exists(DATA_PATH):
    for d in os.listdir("/kaggle/input"):
        c = os.path.join("/kaggle/input", d)
        if os.path.isdir(c) and (os.path.exists(os.path.join(c, "test_in")) or os.path.exists(os.path.join(c, "raw"))):
            DATA_PATH = c; break
        if os.path.isdir(c):
            for s in os.listdir(c):
                sc = os.path.join(c, s)
                if os.path.isdir(sc) and os.path.exists(os.path.join(sc, "test_in")):
                    DATA_PATH = sc; break
            if os.path.exists(os.path.join(DATA_PATH, "test_in")): break
print(f"DATA_PATH: {DATA_PATH}")

# ==================== MODEL PATHS ====================
MODEL_DIR = "/kaggle/input/datasets/saisaaho/v10-models"
MODEL_PATHS = [
    os.path.join(MODEL_DIR, "v10_model_0.pt"),
    os.path.join(MODEL_DIR, "v10_model_1.pt"),
    os.path.join(MODEL_DIR, "v10_model_2.pt"),
]
# Verify all exist
for p in MODEL_PATHS:
    assert os.path.exists(p), f"Model not found: {p}"
print(f"Models: {MODEL_PATHS}")

# ==================== CONFIG ====================
INPUT_STEPS = 10; OUTPUT_STEPS = 16; N_FEAT = 11
TARGET = "cpm25"
MET_FEATURES = ["q2", "t2", "u10", "v10", "swdown", "pblh", "psfc", "rain"]
ALL_MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]
LOG_PM_MAX = math.log1p(500.0)

ENSEMBLE_CONFIGS = [
    {"hid": 64, "drop": 0.1},
    {"hid": 96, "drop": 0.15},
    {"hid": 80, "drop": 0.2},
]

# ==================== COMPUTE NORMALIZATION STATS ====================
print("\n--- Computing normalization stats ---")
base = os.path.join(DATA_PATH, "raw")
stats = {}

for feat in MET_FEATURES:
    arrs = [np.load(os.path.join(base, m, f"{feat}.npy")).astype(np.float32) for m in ALL_MONTHS]
    combined = np.concatenate(arrs).ravel()
    valid = combined[~np.isnan(combined)]
    p1, p99 = np.percentile(valid, [1, 99])
    c = np.clip(valid, p1, p99)
    stats[feat] = {"mean": float(np.mean(c)), "std": float(np.std(c)) + 1e-6}
    print(f"  {feat:10s}: μ={stats[feat]['mean']:.4f}, σ={stats[feat]['std']:.4f}")

u = np.concatenate([np.load(os.path.join(base, m, "u10.npy")).astype(np.float32) for m in ALL_MONTHS])
v = np.concatenate([np.load(os.path.join(base, m, "v10.npy")).astype(np.float32) for m in ALL_MONTHS])
ws = np.sqrt(u**2 + v**2)
stats["wind_speed"] = {"mean": float(np.mean(ws)), "std": float(np.std(ws)) + 1e-6}
stats["wind_dir"] = {"mean": 0.0, "std": float(np.pi)}
print(f"  wind_speed: μ={stats['wind_speed']['mean']:.4f}, σ={stats['wind_speed']['std']:.4f}")
print(f"  wind_dir  : μ=0.0, σ=π")
del u, v, ws

pm = np.concatenate([np.load(os.path.join(base, m, "cpm25.npy")).astype(np.float32) for m in ALL_MONTHS])
pm_log = np.log1p(np.clip(pm, 0, None))
GW_LOG_MEAN_NP = np.mean(pm_log, axis=0)
GW_LOG_STD_NP = np.std(pm_log, axis=0) + 1e-6
gm, gs = float(np.mean(pm_log)), float(np.std(pm_log)) + 1e-6
low = GW_LOG_STD_NP < 0.1
GW_LOG_MEAN_NP[low] = gm
GW_LOG_STD_NP[low] = gs
print(f"  log1p(cpm25): fallback={low.sum()}")
del pm, pm_log
gc.collect()
print("✅ Stats computed")

# ==================== MODEL DEFINITION ====================
class ConvLSTMCell(nn.Module):
    def __init__(self, ind, hid, k=3):
        super().__init__()
        self.hid = hid
        self.conv = nn.Conv2d(ind + hid, 4 * hid, k, padding=k // 2)

    def forward(self, x, state):
        h, c = state
        g = self.conv(torch.cat([x, h], dim=1))
        i, f, o, g = torch.split(g, self.hid, dim=1)
        c_n = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h_n = torch.sigmoid(o) * torch.tanh(c_n)
        return h_n, c_n


class UNetConvLSTM(nn.Module):
    def __init__(self, in_ch=11, out_steps=16, hid=64, drop=0.1):
        super().__init__()
        self.hid = hid
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, hid // 2, 3, padding=1), nn.SELU())
        self.down1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(hid // 2, hid, 3, padding=1), nn.SELU(), nn.Dropout2d(drop))
        self.down2 = nn.MaxPool2d(2)
        self.lstm = ConvLSTMCell(hid, hid, k=3)
        self.up1 = nn.ConvTranspose2d(hid, hid // 2, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(hid, hid // 2, 3, padding=1), nn.SELU(), nn.Dropout2d(drop))
        self.up2 = nn.ConvTranspose2d(hid // 2, hid // 4, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(hid // 4 + hid // 2, out_steps, 3, padding=1))
        self.persistence_gain = nn.Parameter(torch.ones(out_steps))

    def forward(self, x):
        B, T, C, H, W = x.shape
        h_s = torch.zeros(B, self.hid, H // 4, W // 4, device=x.device)
        c_s = torch.zeros(B, self.hid, H // 4, W // 4, device=x.device)

        for t in range(T):
            e1 = self.enc1(x[:, t])
            d1 = self.down1(e1)
            e2 = self.enc2(d1)
            d2 = self.down2(e2)
            h_s, c_s = self.lstm(d2, (h_s, c_s))

        u1 = self.up1(h_s)
        out1 = self.dec1(torch.cat([u1, d1], dim=1))
        u2 = self.up2(out1)
        out = self.dec2(torch.cat([u2, e1], dim=1))

        last_pm = x[:, -1, 0:1]
        return out + last_pm * self.persistence_gain.view(1, -1, 1, 1)


# ==================== LOAD TEST DATA ====================
print("\n--- Loading test data ---")
test_path = os.path.join(DATA_PATH, "test_in")
td = {}
for feat in [TARGET] + MET_FEATURES:
    fp = os.path.join(test_path, f"{feat}.npy")
    if os.path.exists(fp):
        td[feat] = np.load(fp, mmap_mode="r")
        print(f"  ✅ {feat}: {td[feat].shape}")
    else:
        print(f"  ⚠️ {feat} NOT FOUND, using zeros")
        ref = np.load(os.path.join(test_path, "cpm25.npy"), mmap_mode="r")
        td[feat] = np.zeros_like(ref)

nsamples = td["cpm25"].shape[0]
print(f"\nTest samples: {nsamples}")
print(f"Expected output: ({nsamples}, 140, 124, {OUTPUT_STEPS})")

# ==================== INFERENCE ====================
print(f"\n{'='*60}")
print(f"INFERENCE — 3-model ensemble")
print(f"{'='*60}")

NUM_ENSEMBLE = len(MODEL_PATHS)
ens_preds = np.zeros((nsamples, 140, 124, OUTPUT_STEPS), dtype=np.float32)

for ei, path in enumerate(MODEL_PATHS):
    cfg = ENSEMBLE_CONFIGS[ei]
    print(f"\nModel {ei+1}/{NUM_ENSEMBLE}: {os.path.basename(path)} (hid={cfg['hid']}, drop={cfg['drop']})")

    model = UNetConvLSTM(in_ch=11, out_steps=OUTPUT_STEPS, hid=cfg["hid"], drop=cfg["drop"]).to(device)
    model.load_state_dict(torch.load(path, weights_only=True))
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,}")
    print(f"  Persistence gains: [{', '.join(f'{g:.2f}' for g in model.persistence_gain.detach().cpu().numpy())}]")

    bs = 16
    for s in range(0, nsamples, bs):
        e = min(s + bs, nsamples)
        cb = e - s

        x_np = np.zeros((cb, INPUT_STEPS, N_FEAT, 140, 124), dtype=np.float32)

        for b, i in enumerate(range(s, e)):
            for t in range(INPUT_STEPS):
                ch = 0

                # Channel 0: log1p(cpm25) — grid-wise normalized
                raw_pm = np.clip(td[TARGET][i][t].astype(np.float32), 0, None)
                x_np[b, t, ch] = (np.log1p(raw_pm) - GW_LOG_MEAN_NP) / GW_LOG_STD_NP
                ch += 1

                # Channels 1-8: met features — global z-score
                for feat in MET_FEATURES:
                    x_np[b, t, ch] = (td[feat][i][t].astype(np.float32) - stats[feat]["mean"]) / stats[feat]["std"]
                    ch += 1

                # Channel 9: wind_speed — COMPUTED from u10, v10
                u = td["u10"][i][t].astype(np.float32)
                v = td["v10"][i][t].astype(np.float32)
                x_np[b, t, ch] = (np.sqrt(u**2 + v**2) - stats["wind_speed"]["mean"]) / stats["wind_speed"]["std"]
                ch += 1

                # Channel 10: wind_dir — COMPUTED from u10, v10
                x_np[b, t, ch] = (np.arctan2(v, u) - stats["wind_dir"]["mean"]) / stats["wind_dir"]["std"]

        with torch.no_grad(), torch.amp.autocast('cuda'):
            pred = model(torch.from_numpy(x_np).to(device)).float()

        # Denormalize: log-normalized → log1p → raw PM2.5
        pred_np = pred.cpu().numpy()
        pred_log = np.clip(pred_np * GW_LOG_STD_NP[None, None] + GW_LOG_MEAN_NP[None, None], 0, LOG_PM_MAX)
        pred_raw = np.clip(np.expm1(pred_log), 0, 500)

        # Accumulate: (B, T, H, W) → (B, H, W, T), equal weights
        ens_preds[s:e] += pred_raw.transpose(0, 2, 3, 1) / NUM_ENSEMBLE

        if s % 100 == 0:
            print(f"  {s}/{nsamples}")

    del model
    torch.cuda.empty_cache()
    gc.collect()

# ==================== SAVE ====================
ens_preds = np.clip(ens_preds, 0, 500)
np.save("/kaggle/working/preds.npy", ens_preds.astype(np.float32))

print(f"\n{'='*60}")
print(f"✅ V10 ConvLSTM INFERENCE COMPLETE!")
print(f"  Shape:  {ens_preds.shape}")
print(f"  Range:  [{ens_preds.min():.2f}, {ens_preds.max():.2f}]")
print(f"  Mean:   {ens_preds.mean():.2f}")
print(f"  Median: {np.median(ens_preds):.2f}")
print(f"{'='*60}")

Device: cuda
DATA_PATH: /kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2
Models: ['/kaggle/input/datasets/saisaaho/v10-models/v10_model_0.pt', '/kaggle/input/datasets/saisaaho/v10-models/v10_model_1.pt', '/kaggle/input/datasets/saisaaho/v10-models/v10_model_2.pt']

--- Computing normalization stats ---
  q2        : μ=0.0115, σ=0.0070
  t2        : μ=291.6109, σ=13.9322
  u10       : μ=1.5440, σ=3.4679
  v10       : μ=0.1306, σ=2.8897
  swdown    : μ=221.3180, σ=308.1456
  pblh      : μ=759.4549, σ=613.7186
  psfc      : μ=87994.2661, σ=17628.4709
  rain      : μ=0.0556, σ=0.2546
  wind_speed: μ=4.1789, σ=2.5510
  wind_dir  : μ=0.0, σ=π
  log1p(cpm25): fallback=0
✅ Stats computed

--- Loading test data ---
  ✅ cpm25: (218, 10, 140, 124)
  ✅ q2: (218, 10, 140, 124)
  ✅ t2: (218, 10, 140, 124)
  ✅ u10: (218, 10, 140, 124)
  ✅ v10: (218, 10, 140, 124)
  ✅ swdown: (218, 10, 140, 124)
  ✅ pblh: (218, 10, 140, 124)
  ✅ psfc: (218, 10, 140,